# Fusion H6a — evaluator demo

Open in Kaggle, join the competition under your own account, and attach the official competition dataset. Use **Run All** in an interactive session to resolve the Kaggle assets for the selected mode automatically before using Save Version. Internet is needed to clone the public repository when opening this notebook alone, and to install any missing packages.

- **verified_replay** (CPU): uses the original four-view and S0 predictions and recomputes H6a. The original archived replay produced the champion bytes. This streamlined launcher is newly released.
- **fresh_s0** (GPU): runs the ten S0 U-Nets on original test photos and uses the original Mul2 receipts. Fresh S0 parity is unverified. Export requires the exact champion digest.
- **full_inference** (CUDA, preferably dual T4): rebuilds both branches from official test images and signed weights. It uses G3/Q1 recipe metadata and the frozen upstream quality model. Archived predictions are not inputs. This path awaits real GPU and 180-photo checksum validation.

## 1. Environment and automatic asset acquisition

The launcher checks the environment and uses KaggleHub for pinned dataset versions. No API key is embedded. The competition data are read from Kaggle's official mount only. See the [asset inventory](https://github.com/wangzgui/solar-filament-segmentation-2026/blob/main/docs/assets.md) for the exact models, checkpoint names, cached predictions, and provenance checks.

In [ ]:
from pathlib import Path
import subprocess
import sys

MODE = "full_inference"  # or "verified_replay" (CPU) / "fresh_s0" (GPU)
SMOKE_PHOTOS = 0          # use 1 or 2 for a raw-image diagnostic
COMPETITION_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/solar_final_demo")
REPO = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/final_demo.py").is_file()),
    Path("/kaggle/working/solar-filament-segmentation-2026"),
)
if not (REPO / "src/final_demo.py").is_file():
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/wangzgui/solar-filament-segmentation-2026.git", str(REPO)],
        check=True,
    )
sys.path.insert(0, str(REPO / "src"))
from final_demo import run

## 2. Preprocessing and neural predictions

A1 uses a 1536-pixel Mask R-CNN with a ResNet-50 FPN. V1 uses a five-channel ResNet18 U-Net refiner. The original Mul2 code applies two flips and two quarter turns, restores masks to the native grid, and produces a disjoint vote baseline plus candidates. The separate S0 ensemble uses disc-normalized inputs and eight test-time views per checkpoint. [Read the source and weight map](https://github.com/wangzgui/solar-filament-segmentation-2026/blob/main/docs/assets.md).

## 3. Contour correction, quality scoring and final selection

Compute F4 associations from Mul2 and semantic components, adjust local boundaries, rebuild candidate features, apply the frozen Ridge coefficients and run H6a's existing weak-reference risk adjustment and constrained set selection. Serialize the masks to COCO RLE and accept the output only if it matches the signed champion.

In [ ]:
submission = run(MODE, COMPETITION_ROOT, OUTPUT_ROOT, smoke_photos=SMOKE_PHOTOS)
if not SMOKE_PHOTOS:
    subprocess.run([sys.executable, str(REPO / "verify_submission.py"), str(submission)], check=True)
print(submission)

## 4. Optional training and labeled evaluation

Training is skipped in the evaluator run. Open the original notebook in this same checkout to use the indicated entry point. Do not overwrite the signed model archives with newly trained weights.

| Stage | Implementation |
| --- | --- |
| V1 detector/refiner training from the documented initialization | [01_v1_training.ipynb](01_v1_training.ipynb) |
| A1 detector adaptation and checkpoint inference | [02_a1_detector.ipynb](02_a1_detector.ipynb) |
| Mul2 four-view inference with original A1 inputs | [03_mul2_multiview.ipynb](03_mul2_multiview.ipynb) |
| Afshar S0 weight loading, preprocessing and inference | [04_s0_semantic.ipynb](04_s0_semantic.ipynb) |
| Historical development/audit evaluation with the official training annotations | [05_h6a_fusion.ipynb](05_h6a_fusion.ipynb) |
| Original Blended model training | [06_blended_training.ipynb](06_blended_training.ipynb) |
| Original G3 and Q1 intermediate producers | [07_g3_graph.ipynb](07_g3_graph.ipynb) / [08_q1_quality.ipynb](08_q1_quality.ipynb) |

The Afshar ensemble is an external trained model and is loaded using the signed author weights. The release contains our training workflows and the external model's architecture/inference; it does not claim to recreate Afshar's training. [Reproduction guide](https://github.com/wangzgui/solar-filament-segmentation-2026/blob/main/docs/reproduction.md).